# 03. 홈페이지 본문으로 기업 설명 보강

소개가 부족한 기업의 검색 입력을 보강하는 선택 단계입니다. 제공된 웹사이트 본문 외의 정보를 사용하지 않고, 생성 결과는 항상 검토 전 상태로 저장합니다.

> 역사적 매칭 결과는 2025년에 당시 Gemini 1.5 Flash 기반으로 생성됐습니다. 해당 모델은 종료되었으므로, 아래 코드는 현행 Google GenAI SDK를 사용하는 **공개 참조 구현**입니다. 역사적 결과 문장을 동일하게 재현하는 용도로 사용하면 안 됩니다.

In [ ]:
from pathlib import Path
import os
import time

import pandas as pd
from google import genai
from google.genai import types
from tqdm.auto import tqdm

INPUT_PATH = Path('../data/raw/company_crawling.csv')
OUTPUT_PATH = Path('../artifacts/company_identity.csv')
TEXT_COLUMN = 'crawling_result'
API_KEY = os.environ.get('GOOGLE_API_KEY')
MODEL_NAME = os.environ.get('GEMINI_MODEL', 'gemini-3.5-flash')

if not API_KEY:
    raise RuntimeError('Set GOOGLE_API_KEY before running this notebook.')
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Private input is not available: {INPUT_PATH}')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
client = genai.Client(api_key=API_KEY)

In [ ]:
def get_company_identity(crawling_text: str) -> str:
    prompt = f'''
당신은 기업 웹사이트 본문을 읽고 핵심 사업과 기술을 한 문장으로 정리하는 분석가입니다.
제공된 본문 이외의 외부 정보나 도구를 사용하지 마세요.
본문만으로 기업 정체성을 판단하기 어렵다면 반드시 '분석 불가'라고만 답하세요.

[웹사이트 본문]
{crawling_text}

[한 줄 기업 소개]
'''
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2),
    )
    return (response.text or '분석 불가').strip() or '분석 불가'

In [ ]:
company_df = pd.read_csv(INPUT_PATH)
if TEXT_COLUMN not in company_df.columns:
    raise KeyError(f'Missing required column: {TEXT_COLUMN}')

result_df = company_df.copy()
result_df['company_identity'] = ''
result_df['company_identity_needs_review'] = True
result_df['company_identity_status'] = 'pending'

for index, crawling_text in tqdm(result_df[TEXT_COLUMN].items(), total=len(result_df)):
    text = '' if pd.isna(crawling_text) else str(crawling_text).strip()
    if not text:
        result_df.at[index, 'company_identity_status'] = 'missing_crawling_text'
        continue

    result_df.at[index, 'company_identity'] = get_company_identity(text)
    result_df.at[index, 'company_identity_status'] = 'generated'
    time.sleep(2)

result_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
result_df[['company_identity', 'company_identity_status', 'company_identity_needs_review']].head()